In [0]:
%pip install shap

In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# Load gold_county_yield_climate table
gold = spark.table("workspace.default.gold_county_yield_climate")
df = gold.toPandas()

# Choose weather/irrigation feature columns and yield target
feature_cols = [
    'rain_apr_sep_mm', 'daytemp_apr_sep_c', 'nighttemp_apr_sep_c', 'rh_apr_sep_pct',
    'longest_frost_free_days', 'corn_frost_free_ok_int', 'stations_used', 'irrigation', 'year'
]
target_col = 'yield'
# Drop rows with nulls
categorical = ['irrigation']
df = df.dropna(subset=[target_col] + feature_cols)

# Encode categorical feature
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(df[categorical])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical), index=df.index)

num_features = df[[
    'rain_apr_sep_mm', 'daytemp_apr_sep_c', 'nighttemp_apr_sep_c', 'rh_apr_sep_pct',
    'longest_frost_free_days', 'corn_frost_free_ok_int', 'stations_used', 'year'
]]
X = pd.concat([num_features, encoded_df], axis=1)
y = df[target_col]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026)

# Fit XGBoost regressor
model = XGBRegressor(tree_method='hist', random_state=2026)
model.fit(X_train, y_train)

# Feature importances
importances = model.feature_importances_
features = X.columns
sorted_idx = np.argsort(importances)[::-1]

print("Top features impacting yield (gold_county_yield_climate):")
for i in sorted_idx[:10]:
    print(f"{features[i]}: {importances[i]:.4f}")

plt.figure(figsize=(10, 5))
plt.bar(features[sorted_idx[:10]], importances[sorted_idx[:10]])
plt.title('Top 10 Feature Importances for Yield (XGBoost, gold_county_yield_climate)')
plt.xticks(rotation=45)
plt.ylabel('Importance')
plt.tight_layout()
plt.savefig("/Workspace/Users/demirhanceliik@gmail.com/WeatherYield/feature_importance_weather.png")
print("Feature importance chart saved as 'feature_importance_weather.png' in your Workspace files.")
plt.show()

# Summarize model performance
preds = model.predict(X_test)
print(f"R2 Score (test set): {r2_score(y_test, preds):.3f}")

print("\nExplanation:")
print("Ranking shows which weather and irrigation metrics most affect yield variation. Higher scores mean greater impact on yield according to XGBoost.")

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

# Predict yield with fitted model
predicted_yield = model.predict(X)
residuals = y - predicted_yield
df['residual'] = residuals

# Standardize residuals
df['residual_z'] = zscore(residuals)

# Flag unusually high/low outliers (z > 1.5 or z < -1.5)
outlier_mask = (df['residual_z'] > 1.5) | (df['residual_z'] < -1.5)
outliers = df[outlier_mask]

print("Cases with unusually big or low yield relative to weather predictors:")
if not outliers.empty:
    display(outliers[['county_name', 'year', 'yield', 'residual', 'residual_z']])
else:
    print("No outliers detected.")

# Optional: visualize distribution of residuals
import matplotlib.pyplot as plt
plt.figure(figsize=(9,4))
plt.hist(df['residual_z'], bins=30)
plt.title('Yield residuals (z-score) relative to weather/irrigation model')
plt.xlabel('Standardized Residual')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print("\nOutliers represent counties or years where actual yield is much higher or lower than expected. Review these cases for agronomic investigation.")

In [0]:
import shap
import numpy as np

explainer = shap.Explainer(model)
shap_values = explainer(X)

outlier_inds = outliers.index if not outliers.empty else []
outlier_positions = np.array([X.index.get_loc(idx) for idx in outlier_inds])

print("Leading feature cause for each outlier:")
if len(outlier_positions) > 0:
    for pos in outlier_positions[:10]:
        abs_vals = np.abs(shap_values.values[pos])
        top_feat_idx = np.argmax(abs_vals)
        print(f"County: {df.iloc[pos]['county_name']} | Year: {df.iloc[pos]['year']} | Residual: {df.iloc[pos]['residual']:.2f} | Most influencing feature: {X.columns[top_feat_idx]} | SHAP value: {shap_values.values[pos][top_feat_idx]:.4f}")
    if len(outlier_positions) > 1:
        shap.summary_plot(shap_values.values[outlier_positions], X.iloc[outlier_positions], plot_type='bar')
    elif len(outlier_positions) == 1:
        shap.summary_plot(np.array([shap_values.values[outlier_positions[0]]]), X.iloc[[outlier_positions[0]]], plot_type='bar')
else:
    print("No outlier cases found for SHAP attribution.")

print("\nFor each outlier, the feature with the largest SHAP impact is shown as the leading cause for its unusually high or low yield relative to weather/irrigation predictors.")